In [ ]:
import numpy as np
from scipy.optimize import linprog
import math

# --- ИСХОДНЫЕ ДАННЫЕ (Вариант №2) ---

# Горизонтальные ребра (на восток)
# H[y][x] — стоимость перехода из (x, y) в (x+1, y)
H = np.array([
    [6, 3, 4],  # y = 0
    [7, 8, 6],  # y = 1
    [7, 8, 9],  # y = 2
    [8, 9, 10]  # y = 3
])

# Вертикальные ребра (на север)
# V[y][x] — стоимость перехода из (x, y) в (x, y+1)
V = np.array([
    [4, 6, 5, 9],  # y = 0 -> 1
    [5, 7, 6, 8],  # y = 1 -> 2
    [3, 8, 4, 7]   # y = 2 -> 3
])

nx, ny = 4, 4  # Сетка 4x4 узла (от 0 до 3)

# 1.1 Метод динамического программирования (ДП)
def solve_dp():
    dp = np.zeros((ny, nx))
    dp[0, 0] = 0  # Начальная точка A
    
    # Заполнение границ
    for x in range(1, nx):
        dp[0, x] = dp[0, x-1] + H[0, x-1]
    for y in range(1, ny):
        dp[y, 0] = dp[y-1, 0] + V[y-1, 0]
        
    # Основной цикл ДП
    for y in range(1, ny):
        for x in range(1, nx):
            from_west = dp[y, x-1] + H[y, x-1]
            from_south = dp[y-1, x] + V[y-1, x]
            dp[y, x] = min(from_west, from_south)
            
    return dp[ny-1, nx-1]

# 1.2 Жадный алгоритм
def solve_greedy():
    x, y = 0, 0
    total_cost = 0
    while x < nx-1 or y < ny-1:
        if x == nx-1: 
            total_cost += V[y, x]
            y += 1
        elif y == ny-1:
            total_cost += H[y, x]
            x += 1
        else:
            # Выбор локально лучшего шага
            if H[y, x] < V[y, x]:
                total_cost += H[y, x]
                x += 1
            else:
                total_cost += V[y, x]
                y += 1
    return total_cost

# 1.3 Линейное программирование (через scipy)
def solve_lp():
    # Всего 24 ребра (12 гориз. + 12 вертик.) = 24 переменные
    c = np.concatenate([H.flatten(), V.flatten()])
    
    # 16 узлов = 16 уравнений баланса потока
    A_eq = np.zeros((nx*ny, 24))
    b_eq = np.zeros(nx*ny)
    b_eq[0] = 1          # Исток (узел 0,0)
    b_eq[nx*ny - 1] = -1 # Сток (узел 3,3)
    
    # Заполняем коэффициенты: +1 для выходящего потока, -1 для входящего
    # Горизонтальные ребра (индексы 0-11 в векторе c)
    for y in range(ny):
        for x in range(nx-1):
            edge_idx = y * (nx-1) + x
            A_eq[y*nx + x, edge_idx] = 1      # выход из (x, y)
            A_eq[y*nx + x + 1, edge_idx] = -1  # вход в (x+1, y)
            
    # Вертикальные ребра (индексы 12-23 в векторе c)
    for y in range(ny-1):
        for x in range(nx):
            edge_idx = 12 + y * nx + x
            A_eq[y*nx + x, edge_idx] = 1      # выход из (x, y)
            A_eq[(y+1)*nx + x, edge_idx] = -1  # вход в (x, y+1)

    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=(0, 1), method='highs')
    return res.fun


def solve_task_2():
    # 1. ДП для подсчета путей
    counts = np.zeros((ny, nx), dtype=int)
    counts[0, :] = 1
    counts[:, 0] = 1
    for y in range(1, ny):
        for x in range(1, nx):
            counts[y, x] = counts[y-1, x] + counts[y, x-1]
    
    # 2. Комбинаторная формула C(n+m, n)
    # Путь из (0,0) в (3,3) — это 3 шага вправо и 3 шага вверх. Всего 6 шагов.
    n_steps_x = nx - 1
    n_steps_y = ny - 1
    comb_res = math.comb(n_steps_x + n_steps_y, n_steps_x)
    
    return counts[-1, -1], comb_res


dp_val = solve_dp()
greedy_val = solve_greedy()
lp_val = solve_lp()
paths_dp, paths_comb = solve_task_2()

print(f"РЕЗУЛЬТАТЫ ЗАДАНИЯ 1 (Вариант 2)")
print(f"Минимальная стоимость (метод ДП): {dp_val}")
print(f"Стоимость (жадный алгоритм): {greedy_val}")
print(f"Минимальная стоимость (симплекс-метод ЛП): {lp_val}")
print(f"Вывод: {'Жадный алгоритм не оптимален' if greedy_val > dp_val else 'Жадный алгоритм нашел оптимум'}")

print(f"\nРЕЗУЛЬТАТЫ ЗАДАНИЯ 2")
print(f"Количество путей (через ДП): {paths_dp}")
print(f"Количество путей (формула сочетаний): {paths_comb}")
print(f"Оценка сложности алгоритма: O(n * m)")

РЕЗУЛЬТАТЫ ЗАДАНИЯ 1 (Вариант 2)
Минимальная стоимость (метод ДП): 34.0
Стоимость (жадный алгоритм): 39
Минимальная стоимость (симплекс-метод ЛП): 34.0
Вывод: Жадный алгоритм не оптимален

РЕЗУЛЬТАТЫ ЗАДАНИЯ 2
Количество путей (через ДП): 20
Количество путей (формула сочетаний): 20
Оценка сложности алгоритма: O(n * m)
